# Prática — Módulo 10 - Embeddings Semânticos (SBERT) + KMeans

In [1]:
# Carregar dados
import pandas as pd

df = pd.read_parquet("/content/df_final_1024.parquet")

print("Shape:", df.shape)

Shape: (736, 10)


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736 entries, 0 to 735
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   ClienteCod             736 non-null    int64 
 1   ConteudoCategoriaNome  736 non-null    object
 2   ConteudoNome           736 non-null    object
 3   ConteudoDescricao      736 non-null    object
 4   qtd_conteudos          736 non-null    int64 
 5   nome_limpo             736 non-null    object
 6   desc_limpa             736 non-null    object
 7   categorias_limpas      736 non-null    object
 8   texto_final            736 non-null    object
 9   cluster                736 non-null    int32 
dtypes: int32(1), int64(2), object(7)
memory usage: 54.8+ KB


In [3]:
df.head()

,ClienteCod,ConteudoCategoriaNome,ConteudoNome,ConteudoDescricao,qtd_conteudos,nome_limpo,desc_limpa,categorias_limpas,texto_final,cluster
0,68491469,Concurso Público,Curso completo Polícia Militar de Sergipe,Curso preparatório completo em PDF e vídeo aul...,1,polícia militar sergipe,preparatório pdf vídeo aulas concurso polícia ...,[concurso público],concurso público,12
1,14400603,Vendas,Elementária - INGRESSO STANDART,"Elementária é um evento, não um curso. Um marc...",1,elementária ingresso standart,elementária evento marco empresários empresári...,[vendas],vendas,0
2,32136775,Artesanato; Empreendedorismo; Artes; Negócios ...,A MAGIA DA SABOARIA ARTESANAL (INICIANTE),A MAGIA DA SABOARIA ARTESANAL\n\nUma nova jorn...,1,magia saboaria artesanal iniciante,magia saboaria artesanal nova jornada começa a...,"[artesanato, empreendedorismo, artes, negócios...",artesanato empreendedorismo artes negócios e d...,29
3,89645602,"Saúde; Saúde, dieta e beleza",Como diminuir seu colesterol,?? Produto: eBook – Como Diminuir o Colesterol...,1,diminuir colesterol,produto diminuir colesterol naturalmente trans...,"[saúde, saúde, dieta e beleza]","saúde saúde, dieta e beleza",1
4,94688887,"Marketing; Vendas; Software, TI e Internet; We...","Crie GRATUITAMENTE Vídeos, Imagens e Sons que ...","Criação de Mídias que Vendem – Imagens, Vídeos...",1,crie gratuitamente vídeos imagens sons vendem ...,criação mídias vendem imagens vídeos sons pequ...,"[marketing, vendas, software, ti e internet, w...","marketing vendas software, ti e internet web d...",17


# **Fase 0 - Engenharia de Atributos**

In [4]:
# Transformar cada texto em lista de tokens
import re

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-záéíóúâêôãõç\s]", " ", text)
    return text.split()

sentences = df["texto_final"].apply(tokenize).apply(lambda x: x[:300])

# **Fase 1 - Representação SBERT**

In [5]:
# Criar campo textual único para vetorização
# Determinar qual campo textual vale manter
df["texto_final"] = (
    df["categorias_limpas"].apply(lambda x: " ".join(x))  # <- lista de palavras
    # + " " + df["nome_limpo"]
    + " " + df["desc_limpa"]
).str.strip()

In [6]:
# Treinar embeddings de palavras com Embeddings de Sentença (SBERT)
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

textos = df["texto_final"]

X = model.encode(
    textos.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Shape:", X.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Shape: (736, 384)


# **Fase 2 - Clustering (KMeans)**

In [7]:
# Agrupar conteúdos usando KMeans
from sklearn.cluster import KMeans

k = 12

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

## Validação

### Silhouette Score

In [8]:
# Calcular qualidade geométrica dos clusters
from sklearn.metrics import silhouette_score

score = silhouette_score(X, df["cluster"])
print("Silhouette:", score)

Silhouette: 0.054887585


### Visualização de Clusters

In [9]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

for i in sorted(df["cluster"].unique()):
    print(f"\nCluster {i}:")

    exemplos = df[df["cluster"] == i]["texto_final"]

    n = min(4, len(exemplos))
    for t in exemplos.sample(n, random_state=42):
      print(t)


Cluster 0:
coaching capacitação ingresso trabalho estudantes formação recém formados física encontros on line trabalho alavancar mentalidade empreendedora participante além disso irei ensinar metodologia baseada funcional dividida módulos diferentes final semana mês durante meses
educação física e esporte nesse receberá atendimento personalizado treinos direcionados tratar dor alívio recuperação completa nesse receberá tratamento personalizado suporte caso acompanharemos todas etapas alívio recuperação completa
educação física e esporte saúde sente dificuldades hora alongar treinos parece fazendo algo errado então book saiba executar principais alongamentos detalhada assertiva
saúde, dieta e beleza protocolo deusa sheipada mulheres desejam melhorar treinos impulsionar resultados academia alcançando corpo definido digno verdadeira deusa dicas práticas treino aliado atingir objetivos academia sentir verdadeira deusa agora atualizado nova versão traz ainda eficazes ajustes treinos dicas 

In [10]:
# Ver exemplos de separação
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(
    df.groupby("cluster")["texto_final"]
    .apply(lambda x: x.sample(min(len(x), 4), random_state=42))
)

cluster     
0        163                                                                                                                                                                            coaching capacitação ingresso trabalho estudantes formação recém formados física encontros on line trabalho alavancar mentalidade empreendedora participante além disso irei ensinar metodologia baseada funcional dividida módulos diferentes final semana mês durante meses
         691                                                                                                                                                                                                              educação física e esporte nesse receberá atendimento personalizado treinos direcionados tratar dor alívio recuperação completa nesse receberá tratamento personalizado suporte caso acompanharemos todas etapas alívio recuperação completa
         715                                                                                                                                                                                                                                                                                 educação física e esporte saúde sente dificuldades hora alongar treinos parece fazendo algo errado então book saiba executar principais alongamentos detalhada assertiva
         689                                                                                                                       saúde, dieta e beleza protocolo deusa sheipada mulheres desejam melhorar treinos impulsionar resultados academia alcançando corpo definido digno verdadeira deusa dicas práticas treino aliado atingir objetivos academia sentir verdadeira deusa agora atualizado nova versão traz ainda eficazes ajustes treinos dicas maximizar
1        453                                                                                                                                                                                                                                                                                                                                                             área médica neste aprenderá melhores prevenir varizes caso aprenderá tratá las foco melhorar
         641                                                                                                                                    educação pet terapia quarteto fantástico intervenções busca diferencial real currículo deseja entregar resultados únicos pacientes alunos pet terapia versão formação completa intervenções assistidas animais saa focada quarteto fantástico cães aves roedores répteis desenhado profissionais estudantes áreas saú
         419                                                                                                                                                                                                                                                                                                                                      saúde todo ensinamento necessário deseja tratar descobrir possui lipedema saber identificar estágios dessa condição
         670                                                                                                                                      vendas laserterapia desenvolvido capacitar profissionais saúde todas etapas atendimento laser baixa intensidade embasamento teórico fundamentos científicos passando elaboração plano terapêutico individualizado uso termo consentimento aborda protocolos aplicação seguras dicas marketing impulsionar atuação e
2        292                                                                                                                                                                                                           autoajuda e desenv. pessoal beleza e estética conquistar coragem ver medo padrão permissão existir merece sentir linda agora validação medos